# Axis 5 — Memory Hierarchy & Expert Placement (Mac / MLX)

**Question (team page).** How does placing experts across memory tiers affect end-to-end latency and throughput?

**Tiers.** Windows has three tiers (VRAM → RAM → disk). Apple Silicon has **two**: **unified memory** (where the GPU computes from) and **disk** (the SSD). So on the Mac the question becomes: *given a fixed unified-memory budget for experts, which experts should live there, and should that set be fixed or change over time?*

## Strategies compared at the same budget B (experts in unified memory)
| Strategy | Unified-memory tier | Changes over time? |
|---|---|---|
| **static** | the B − 64 experts used most in a *profiling* run, pinned for good, plus a 64-slot staging area (the minimum to compute any layer call) | almost not at all |
| **hybrid** | B/2 pinned hottest experts + B/2 LRU slots | half |
| **dynamic** | B LRU slots, nothing pinned | fully |

* Profiling uses a **different** set of prompts from the evaluation, so pinning is not tuned to the test.
* Pinned experts are loaded before the run (startup cost reported separately, not counted as misses).
* Everything that is not resident is read from disk on demand, page cache bypassed.

**Hypothesis, translated.** A hierarchy that keeps frequently reused experts in the fast tier and fetches only cold tails from disk beats a single uniform policy, and the right split minimises disk reads at a given budget.

**Runtime.** ≈ 45–60 min full; `MOE_SMOKE=1` a few minutes.

In [ ]:
import os, sys, time, json, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlx.core as mx

# Find the repo and import the shared module (experiments/moe_cache.py).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments' / 'moe_cache.py').exists())
sys.path.insert(0, str(ROOT / 'experiments'))
import moe_cache as mc
importlib.reload(mc)

# MOE_SMOKE=1 runs a tiny configuration end to end (used to test the notebook).
SMOKE = os.environ.get('MOE_SMOKE') == '1'
AXIS_DIR = ROOT / 'experiments' / 'axis-5-memory-tiering'
RESULTS = (ROOT / 'experiments' / '.cache' / 'smoke' / 'axis-5-memory-tiering') if SMOKE else (AXIS_DIR / 'results')
RESULTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.figsize': (11, 5), 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})
print('SMOKE run' if SMOKE else 'Full run', '| results ->', RESULTS.relative_to(ROOT))
print(mc.system_info())

## Configuration

In [ ]:
BUDGETS = [192, 384, 576]          # experts resident in unified memory (bank = B x 12.6 MB)
STAGING = 64                       # dynamic slots kept by the static strategy
HYBRID_PIN_FRACTION = 0.5
PROFILE_PER_DOMAIN = 2             # profiling prompts per kind (disjoint from evaluation; max 2+2)
EVAL_PER_DOMAIN = 2
MAX_NEW_TOKENS = 48
REPLAY_BUDGETS = [128, 192, 256, 384, 512, 576, 768]   # denser curve by exact replay

if SMOKE:
    BUDGETS, PROFILE_PER_DOMAIN, EVAL_PER_DOMAIN, MAX_NEW_TOKENS = [192], 1, 1, 6
    REPLAY_BUDGETS = [128, 192, 256]
print('budgets:', BUDGETS, '| strategies: static, hybrid, dynamic')

## Model and correctness gate

In [ ]:
model, tokenizer, index = mc.load_model()
print(index.describe())
print(f'Non-expert weights resident: {mc.non_expert_bytes(model) / 1e9:.2f} GB '
      f'| MLX active memory: {mx.get_active_memory() / 1e9:.2f} GB')
print(f'Largest K that fits safely in this machine\'s RAM: {mc.max_safe_capacity(index)}')

# Correctness gate: one layer through the K-slot bank vs the stock mlx-lm layer
# (all 64 experts of that layer stacked). Must be bit-identical.
diff = mc.check_layer_exactness(model, index, capacity=64)
assert diff == 0.0, diff
print(f'Layer exactness vs stock mlx-lm: max |diff| = {diff}')

## Profiling run: which experts are hot?

In [ ]:
# Written for this axis: per kind of request, the first PROFILE_PER_DOMAIN prompts are used
# to find hot experts and the following EVAL_PER_DOMAIN are the held-out evaluation traffic.
PROMPTS = {
    'math': ['A recipe needs 250 g of flour for 12 cookies. How much flour for 30 cookies?',
             'A bike travels 18 km in 45 minutes. What is its speed in km/h?',
             'What is 7/8 written as a percentage, and how do you convert it?',
             'Three friends split a bill of $84 with a 15% tip. How much does each pay?'],
    'code': ['Write a Python function that removes duplicates from a list while keeping order.',
             'Write a Python function that converts Roman numerals to integers.',
             'Write a Python function that finds the longest common prefix of a list of strings.',
             'Write a Python function that rotates a matrix by 90 degrees.'],
    'chat': ['What are some tips for sleeping better?',
             'How does a credit score work?',
             'What is the best way to learn a new language as an adult?',
             'Why is the sky blue?'],
    'writing': ['Write a short story about a cat who runs a bakery.',
                'Write a haiku sequence about a city waking up.',
                'Write a short diary entry from an astronaut on the first day in orbit.',
                'Write a cheerful birthday message for a grandmother turning ninety.'],
}
profile_reqs = mc.requests(PROMPTS, n_per_domain=PROFILE_PER_DOMAIN, order='interleave')
eval_reqs = mc.requests(PROMPTS, n_per_domain=EVAL_PER_DOMAIN, order='interleave', offset=PROFILE_PER_DOMAIN)
assert not {p['text'] for p in profile_reqs} & {p['text'] for p in eval_reqs}

cache = mc.ExpertCache(index, 256, 'LRU')
_, rec = mc.run_workload(model, tokenizer, cache, profile_reqs, MAX_NEW_TOKENS, log=None)
cache.close(); mc.detach_cache(model)
prof = mc.events_frame(rec.events)
ranking = prof.groupby(['layer', 'eid']).size().sort_values(ascending=False)
hot = [tuple(map(int, k)) for k in ranking.index]
print(f'profiling: {len(profile_reqs)} requests, {len(prof):,} accesses, {len(hot)} distinct experts')
print('top-10 experts (layer, id):', hot[:10])
print('share of accesses covered by the top 128 / 256 / 512:',
      [f'{ranking.iloc[:n].sum() / ranking.sum():.0%}' for n in (128, 256, 512)])

## Real runs on the evaluation requests

In [ ]:
def make_strategy(name, B):
    if name == 'dynamic':
        return mc.make_policy('LRU', B), []
    n_pin = B - STAGING if name == 'static' else int(B * HYBRID_PIN_FRACTION)
    pins = hot[:n_pin]
    return mc.TieredPolicy(B, pins, 'LRU'), pins

SAFE_K = mc.max_safe_capacity(index)
rows, traces = [], {}
for B in [b for b in BUDGETS if b <= SAFE_K]:
    for strat in ['static', 'hybrid', 'dynamic']:
        mc.free_memory(); mx.reset_peak_memory()
        pol, pins = make_strategy(strat, B)
        cache = mc.ExpertCache(index, B, pol)
        startup_ms = cache.preload(pins)
        t0 = time.perf_counter()
        df, rec = mc.run_workload(model, tokenizer, cache, eval_reqs, MAX_NEW_TOKENS, log=None)
        s = cache.summary()
        p2, _ = make_strategy(strat, B)
        hits = mc.replay(rec.events, p2)
        assert int(hits.sum()) == s['hits'], 'replay mismatch'
        dec_tok = (df.new_tokens - 1).sum()
        acc = mc.events_frame(rec.events)
        rows.append({'B': B, 'strategy': strat, 'pinned': len(pins), 'hit_rate': s['hit_rate'],
                     'decode_hit_rate': hits[(acc.phase == 'decode').to_numpy()].mean(),
                     'GB_loaded': s['GB_loaded'], 'io_s': s['io_ms'] / 1000,
                     'startup_s': startup_ms / 1000, 'startup_GB': len(pins) * index.expert_bytes / 1e9,
                     'decode_tok_s': dec_tok / df.decode_s.sum(), 'ttft_mean_s': df.ttft_s.mean(),
                     'wall_s': time.perf_counter() - t0, 'peak_GB': mx.get_peak_memory() / 1e9})
        traces[(B, strat)] = rec.events
        cache.close(); mc.detach_cache(model)
        r = rows[-1]
        print(f'B={B} {strat:8s}: hit {r["hit_rate"]:.1%} (decode {r["decode_hit_rate"]:.1%}) | '
              f'{r["GB_loaded"]:.1f} GB read in {r["io_s"]:.1f}s | {r["decode_tok_s"]:.2f} tok/s | '
              f'TTFT {r["ttft_mean_s"]:.2f}s | startup {r["startup_s"]:.1f}s')
res = pd.DataFrame(rows)
res.to_csv(RESULTS / 'axis5_runs.csv', index=False)
res.style.format(precision=2)

## Denser budget curve by exact replay
The evaluation access stream is the same for every strategy (outputs do not depend on the cache), so each strategy can be replayed at more budgets — including ones too large to time on this machine. Replay was asserted equal to every real run above.

In [ ]:
events = next(iter(traces.values()))
acc = mc.events_frame(events)
dec = (acc.phase == 'decode').to_numpy()
rep = []
for B in REPLAY_BUDGETS:
    for strat in ['static', 'hybrid', 'dynamic']:
        if strat == 'static' and B <= STAGING:
            continue
        pol, _ = make_strategy(strat, B)
        h = mc.replay(events, pol)
        rep.append({'B': B, 'strategy': strat, 'hit_rate': h.mean(), 'decode_hit_rate': h[dec].mean(),
                    'GB_read': (~h).sum() * index.expert_bytes / 1e9})
rep = pd.DataFrame(rep)
rep.to_csv(RESULTS / 'axis5_replay.csv', index=False)
rep.pivot(index='B', columns='strategy', values='hit_rate').style.format('{:.1%}').highlight_max(axis=1, color='#c6efce')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for strat, d in rep.groupby('strategy'):
    axes[0].plot(d.B, d.hit_rate, '-o', label=strat)
    axes[1].plot(d.B, d.GB_read, '-o', label=strat)
for strat, d in res.groupby('strategy'):
    axes[2].plot(d.B, d.decode_tok_s, '-o', label=strat)
axes[0].set_title('Hit rate (replay)'); axes[0].set_ylabel('hit rate')
axes[1].set_title('Bytes read from disk (replay)'); axes[1].set_ylabel('GB')
axes[2].set_title('Decode throughput (real runs)'); axes[2].set_ylabel('tokens / s')
for ax in axes:
    ax.set_xlabel('unified-memory budget B (experts)'); ax.legend()
fig.tight_layout(); fig.savefig(RESULTS / 'axis5_tiering.png', dpi=150, bbox_inches='tight'); plt.show()

## Hypothesis check (descriptive)

In [ ]:
for B, d in rep.groupby('B'):
    d = d.set_index('strategy').hit_rate
    print(f'B={B:4d}: best = {d.idxmax():8s} ({d.max():.1%}); ' + ', '.join(f'{k} {v:.1%}' for k, v in d.items()))
for B, d in res.groupby('B'):
    d = d.set_index('strategy')
    print(f'real B={B}: fastest decode = {d.decode_tok_s.idxmax()} ({d.decode_tok_s.max():.2f} tok/s); '
          f'static pays {d.loc["static", "startup_s"]:.1f}s of startup loading '
          f'({d.loc["static", "startup_GB"]:.1f} GB)')
print('\nA tiered layout "wins" here only if pinning what was hot in profiling beats pure recency on new prompts.')